<a href="https://colab.research.google.com/github/shashankcodes1903/Ontology-Project-HPE-/blob/main/hemanthcs-ontology.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

nlp = spacy.load("en_core_web_sm")


# STEP 1:  PREPROCESSING
def preprocess_text(text):
    doc = nlp(text)
    tokens = []

    for token in doc:
        if token.is_stop or token.is_punct or token.like_num:
            continue

        word = token.text.lower().strip()


        if token.pos_ == "PROPN":
            tokens.append(word)

        elif token.pos_ == "NOUN":
            tokens.append(token.lemma_.lower())

        elif token.pos_ == "VERB":
            tokens.append(word)

        else:
            tokens.append(word)

    return tokens





# STEP 2A: TF-IDF

def extract_tfidf_terms(tokens, top_k=10):
    text = " ".join(tokens)

    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform([text])

    feature_names = vectorizer.get_feature_names_out()
    scores = X.toarray()[0]

    term_scores = list(zip(feature_names, scores))
    term_scores.sort(key=lambda x: x[1], reverse=True)

    return term_scores[:top_k]



# STEP 2B: NOUN PHRASES

def extract_noun_phrases(text):
    doc = nlp(text)
    phrases = []

    for chunk in doc.noun_chunks:
        phrases.append(chunk.text.lower().strip())

    return phrases



# STEP 2: COMBINED TERM EXTRACTION

def extract_terms(text, tokens):
    tfidf_terms = extract_tfidf_terms(tokens)
    noun_phrases = extract_noun_phrases(text)

    tfidf_words = [term for term, score in tfidf_terms]

    combined_terms = list(set(tfidf_words + noun_phrases))

    return combined_terms, dict(tfidf_terms)



# STEP 3: CONCEPT FORMATION

def extract_concepts(terms):
    concepts = []

    for term in terms:
        doc = nlp(term)

        for token in doc:
            if token.pos_ in ["NOUN", "PROPN"]:
                concepts.append(token.lemma_.lower())

    return list(set(concepts))


# RELATION EXTRACTION (DEPENDENCY)

def extract_relations(text):
    doc = nlp(text)
    triples = []

    for token in doc:
        if token.pos_ == "VERB":
            subject = None
            obj = None


            for child in token.lefts:
                if child.dep_ in ["nsubj", "nsubjpass"]:
                    subject = child.text


            for child in token.rights:
                if child.dep_ in ["dobj", "attr", "pobj"]:
                    obj = child.text

            if subject and obj:
                triples.append((subject.lower(), token.lemma_.lower(), obj.lower()))

    return triples


# CONFIDENCE SCORING

def compute_confidence(triples, tfidf_scores):
    scored_triples = []

    for subj, rel, obj in triples:
        score_subj = tfidf_scores.get(subj, 0.1)
        score_obj = tfidf_scores.get(obj, 0.1)

        confidence = round((score_subj + score_obj) / 2, 3)

        scored_triples.append((subj, rel, obj, confidence))

    return scored_triples



# FULL PIPELINE

def ontology_pipeline(text):
    print("\n========== INPUT TEXT ==========\n")
    print(text)

    # STEP 1
    tokens = preprocess_text(text)
    print("\n--- STEP 1: CLEAN TOKENS ---")
    print(tokens)

    # STEP 2
    terms, tfidf_scores = extract_terms(text, tokens)
    print("\n--- STEP 2: TERMS ---")
    print(terms)

    # STEP 3
    concepts = extract_concepts(terms)
    print("\n--- STEP 3: CONCEPTS ---")
    print(concepts)

    # RELATIONS
    triples = extract_relations(text)
    print("\n--- RELATIONS (TRIPLES) ---")
    print(triples)

    # CONFIDENCE
    scored_triples = compute_confidence(triples, tfidf_scores)
    print("\n--- TRIPLES WITH CONFIDENCE ---")
    print(scored_triples)

    return {
        "tokens": tokens,
        "terms": terms,
        "concepts": concepts,
        "triples": scored_triples
    }



if __name__ == "__main__":
    text = """
    Bill and Dave became friends when they were both engineering students at Stanford.
    After graduation, Dave took a job with General Electric and moved to Schenectady,
    New York, where he married his college sweetheart Lucile Salter in 1938.
    But he and Bill stayed in touch. The two were encouraged by their former professor Fred Terman
    to start a technology company of their own.
    Taking a leave of absence from his job at GE, Dave and his new bride drove to California with
    a used drill press (an important piece of equipment for the new venture) in the rumble seat.
    Bill scouted for places where the newlyweds could live. He found the ideal rental at 367 Addison
    Avenue in Palo Alto for $45 per month. Dave and Lucile would live in the downstairs flat,
    while Bill would bunk in a tiny backyard shed where there was indoor plumbing and just enough
    room for a cot. But what made the property truly perfect for their needs was the
    small garage that the landlady told them they could use as a workshop.
    """

    ontology_pipeline(text)


========== INPUT TEXT ==========


    Bill and Dave became friends when they were both engineering students at Stanford.
    After graduation, Dave took a job with General Electric and moved to Schenectady,
    New York, where he married his college sweetheart Lucile Salter in 1938. 
    But he and Bill stayed in touch. The two were encouraged by their former professor Fred Terman 
    to start a technology company of their own.
    Taking a leave of absence from his job at GE, Dave and his new bride drove to California with 
    a used drill press (an important piece of equipment for the new venture) in the rumble seat. 
    Bill scouted for places where the newlyweds could live. He found the ideal rental at 367 Addison 
    Avenue in Palo Alto for $45 per month. Dave and Lucile would live in the downstairs flat,
    while Bill would bunk in a tiny backyard shed where there was indoor plumbing and just enough 
    room for a cot. But what made the property truly perfect for their ne